In [23]:
# Custom Initialization 대상 MLP 구성

import torch
from torch import nn

net = nn.Sequential(
    nn.LazyLinear(8), # [2, 4] -> [2, 8]
    nn.ReLU(),        
    nn.LazyLinear(1), # [2, 8] -> [2, 1]
)

# [2, 4]
X = torch.rand(
    size=(2, 4)
)

# Lazy parameter materialization
net(X)

first_layer = net[0]

print("type(first_layer):", type(first_layer))
print("first_layer.weight.shape", first_layer.weight.shape)
print("first_layer.bias.shape", first_layer.bias.shape)

type(first_layer): <class 'torch.nn.modules.linear.Linear'>
first_layer.weight.shape torch.Size([8, 4])
first_layer.bias.shape torch.Size([8])


In [24]:
# define & apply: Custom Initializer

def my_init(
    module: nn.Module,
):
    
    if isinstance(
        module,
        nn.Linear,
    ):
        
        name, parameter = next(
            module.named_parameters()
        )
        
        print(
            "Init",
            name,
            parameter.shape,
        )

        nn.init.uniform_(
            module.weight,
            -10,
            10,
        )

        with torch.no_grad():
            module.weight *= (
                module.weight.abs() >= 5
            )


net.apply(my_init)

print("\n[first_layer.weight]", first_layer.weight)
print("\n[first_layer.bias]", first_layer.bias)


Init weight torch.Size([8, 4])
Init weight torch.Size([1, 8])

[first_layer.weight] Parameter containing:
tensor([[ 0.0000, -0.0000, -9.7989, -0.0000],
        [ 9.5569,  0.0000,  8.1570,  9.3495],
        [ 0.0000, -0.0000, -0.0000,  6.0775],
        [ 7.0979, -7.6818,  6.8347, -0.0000],
        [-0.0000, -8.0006,  9.1678,  0.0000],
        [-0.0000, -0.0000, -7.5904, -0.0000],
        [-0.0000, -0.0000, -6.7166, -6.5822],
        [ 9.4878,  7.3248, -7.7322,  0.0000]], requires_grad=True)

[first_layer.bias] Parameter containing:
tensor([ 0.2433, -0.1294, -0.0140,  0.3170,  0.3559,  0.1399, -0.2075, -0.4684],
       requires_grad=True)


In [ ]:
# access Parameter directly

with torch.no_grad():

    first_layer.weight[:] += 1
    
    first_layer.weight[
        0,
        0,
    ] = 999
    
print("first_layer.weight.shape:", first_layer.weight.shape)
print(
    first_layer.weight
)

first_layer.weight.shape: torch.Size([8, 4])
Parameter containing:
tensor([[999.0000,   1.0000,  -8.7989,   1.0000],
        [ 10.5569,   1.0000,   9.1570,  10.3495],
        [  1.0000,   1.0000,   1.0000,   7.0775],
        [  8.0979,  -6.6818,   7.8347,   1.0000],
        [  1.0000,  -7.0006,  10.1678,   1.0000],
        [  1.0000,   1.0000,  -6.5904,   1.0000],
        [  1.0000,   1.0000,  -5.7166,  -5.5822],
        [ 10.4878,   8.3248,  -6.7322,   1.0000]], requires_grad=True)
